# Stage 2B — CNN training on Colab

ECG Arrhythmia Classification, Paper #2.

This trains the 1D-CNN half of the Stage 2 grid. LR and XGBoost (Part A) were
run on a laptop and are already merged.

**Before you start — put these in Google Drive under `ecg-paper/`:**

```
MyDrive/ecg-paper/
├── src/                 config.py  imbalance.py  models.py  run_grid.py  train_cnn.py
└── artifacts/
    └── ds1.npz          (from the shared Drive folder, ~45 MB)
```

`ds2.npz` is **not** needed and must not be uploaded. DS2 stays sealed until
Stage 4.

**Set the runtime first:** Runtime → Change runtime type → **T4 GPU** → Save.

## 1. Confirm a GPU was actually allocated

Colab sometimes hands out a CPU runtime when GPUs are busy. This cell stops
you rather than letting a run take 8x longer without saying so.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    'No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun this cell.')
print('GPU :', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

## 2. Mount Drive and copy the project in

Work happens on Colab's local disk (fast); Drive is only for getting files in
and results out.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/ecg-paper'
WORK = '/content/work'

!mkdir -p {WORK}/src {WORK}/artifacts
!cp {DRIVE}/src/*.py {WORK}/src/
!cp {DRIVE}/artifacts/ds1.npz {WORK}/artifacts/

%cd {WORK}
!ls -la src/ artifacts/

In [ ]:
# torch and sklearn ship with Colab; these two do not.
!pip install -q xgboost imbalanced-learn

## 3. Sanity check

`ds1.npz` must match what Part A used, or the two halves cannot be merged.
Expect **50,977 beats from 22 patients** and the class counts below.

In [ ]:
import sys
import numpy as np

sys.path.insert(0, 'src')
import config as C

d = np.load('artifacts/ds1.npz')
y, pid = d['y'], d['pid']

print(f'beats    : {len(y):,}   (expect 50,977)')
print(f'patients : {len(np.unique(pid))}       (expect 22)')
print('classes  :', {c: int((y == i).sum()) for i, c in enumerate(C.CLASS_NAMES)})
print('eval on  :', C.EVAL_CLASSES, '(Q excluded)')

assert len(y) == 50977, 'ds1.npz does not match Part A — re-download from Drive'
print('\nOK')

## 4. Smoke test — 10 minutes, do not skip

A subset with few epochs. The numbers are meaningless; you are only checking
that nothing crashes before committing to a multi-hour run.

The predictions it writes are deleted immediately afterwards so a smoke-test
fold cannot leak into real results.

In [ ]:
!python src/run_grid.py --models cnn --strategies none --rr 1 --subset 5000 --epochs 5

In [ ]:
!rm -rf artifacts/preds artifacts/grid_summary.json
print('smoke-test output cleared')

## 5. The real run — 2 to 4 hours

4 strategies × 2 RR settings × 5 folds = **40 fold-runs**.

Each fold is saved the moment it finishes, so a disconnect costs one fold, not
the run. If Colab drops you: rerun cells 1, 2 and this one — completed folds
are skipped automatically.

**Keep this browser tab open and interact with the page every so often.** Colab
disconnects idle sessions after about 90 minutes.

In [ ]:
!python src/run_grid.py --models cnn

## 6. Save results back to Drive

Run this **every 30–40 minutes while the run is going**, not only at the end.
Colab's local disk is wiped on disconnect; Drive is what survives, and these
files are also the resume points.

In [ ]:
!mkdir -p {DRIVE}/artifacts/preds
!cp -r {WORK}/artifacts/preds/* {DRIVE}/artifacts/preds/ 2>/dev/null
!ls {WORK}/artifacts/preds | wc -l
print('^ folds saved to Drive (40 when complete)')

## 7. Read the ablation

Compare `rr=1` against `rr=0` for the same strategy. The gap in the **S**
column is what Stage 2 exists to measure.

Part A found 6/6 pairs improved, S-class F1 between 2.4× and 16× better with
RR features. The CNN should show the same direction.

In [ ]:
import json

with open('artifacts/grid_summary.json') as f:
    rows = json.load(f)

for r in rows:
    pc = r['per_class_f1']
    print(f"{r['model']:4} {r['strategy']:12} rr={int(r['use_rr'])}  "
          f"macroF1 {r['macro_f1_mean']:.4f} +/- {r['macro_f1_std']:.4f}   "
          f"S={pc['S']:.3f} V={pc['V']:.3f} F={pc['F']:.3f}")

## 8. What to expect, and what would be wrong

These are **cross-validation scores inside DS1**, not DS2 test scores. They are
lower and noisier than the published figures, for reasons that are a property of
the database:

* Record **209** holds 383 of DS1's 943 S beats (40%)
* Record **208** holds 372 of its 414 F beats (90%)

No split that keeps patients intact can spread those evenly, so per-fold scores
swing hard and **F is not interpretable in CV at all**. Part A saw per-fold
S-F1 range from 0.03 to 0.78 for the same configuration.

The comparison against published work (SVEB F1 ≈ 68–75) happens in **Stage 4**,
on DS2.

**Worth reporting back if you see it:**

* RR ablation not 6/6 → something is off, say so before anyone builds on it
* macro-F1 above 0.85 → likely a leak, stop and check
* `focal` identical to `none` → the loss is not being applied

## 9. Finishing up

1. Run cell 6 one last time so all 40 folds are on Drive
2. Tell the team the folds are up
3. Commit your work:

```bash
git checkout main && git pull
git checkout -b stage2/cnn
# add src/train_cnn.py and the torch line in requirements.txt
git add src/train_cnn.py requirements.txt
git commit -m "stage2: 1D-CNN with RR fusion, focal loss and early stopping on macro-F1"
git push -u origin stage2/cnn
```

Then open a pull request.

Whoever merges Part A and Part B afterwards combines both `preds/` folders into
one and runs the full grid — every fold file already exists, so it writes the
complete summary in seconds without retraining anything.